# LeetCode 2016 — Maximum Difference Between Increasing Elements bằng Python

**Link đề:** https://leetcode.com/problems/maximum-difference-between-increasing-elements/
**Độ khó:** Easy · **Chủ đề:** Array (tiền tố nhỏ nhất — prefix minimum)

```python
class Solution:
    def maximumDifference(self, nums: List[int]) -> int:

```

## Mục tiêu học
1. Hiểu đề: tìm `nums[j] − nums[i]` lớn nhất với `i < j` **và** `nums[i] < nums[j]` (nghiêm ngặt); không có thì `-1`.
2. Viết vét cạn `O(n²)` hai vòng lặp, thấy rõ công việc lặp lại.
3. Học kỹ thuật **"giữ giá trị nhỏ nhất đã gặp" (running/prefix minimum)** — một lần duyệt `O(n)`, `O(1)` bộ nhớ.
4. Trace từng bước, chứng minh invariant, xử lý đúng trường hợp **bằng nhau** và **giảm dần**.
5. Liên hệ bài kinh điển *Best Time to Buy and Sell Stock* (LeetCode 121), làm bài tập biến thể.

## 1. Đề bài yêu cầu gì?

Cho mảng `nums` có `n` phần tử. Tìm **giá trị lớn nhất** của `nums[j] − nums[i]` sao cho:
- `0 <= i < j < n` (phần tử nhỏ đứng **trước**),
- `nums[i] < nums[j]` (tăng **nghiêm ngặt**).

Nếu không tồn tại cặp nào như vậy, trả `-1`.

| `nums` | Kết quả | Giải thích |
|---|:-:|---|
| `[7,1,5,4]` | `4` | i=1, j=2: 5 − 1. (7 − 1 = 6 không hợp lệ vì 7 đứng **trước** 1) |
| `[9,4,3,2]` | `-1` | dãy giảm dần, không có cặp tăng |
| `[1,5,2,10]` | `9` | i=0, j=3: 10 − 1 |

**Constraints:** `2 <= n <= 1000`, `1 <= nums[i] <= 10^9`.

Chú ý: `[5, 5]` → **−1** (không phải 0) vì `5 < 5` là sai.

In [1]:
nums = [7, 1, 5, 4]
for i in range(len(nums)):
    for j in range(i + 1, len(nums)):
        ok = nums[i] < nums[j]
        print(f"i={i}, j={j}: {nums[j]} - {nums[i]} = {nums[j] - nums[i]:>2}", "(hợp lệ)" if ok else "(không: nums[i] >= nums[j])")

i=0, j=1: 1 - 7 = -6 (không: nums[i] >= nums[j])
i=0, j=2: 5 - 7 = -2 (không: nums[i] >= nums[j])
i=0, j=3: 4 - 7 = -3 (không: nums[i] >= nums[j])
i=1, j=2: 5 - 1 =  4 (hợp lệ)
i=1, j=3: 4 - 1 =  3 (hợp lệ)
i=2, j=3: 4 - 5 = -1 (không: nums[i] >= nums[j])


## 2. Nền tảng Python cần biết

### 2.1. Hai vòng lặp `i < j`

`for j in range(i + 1, n)` sinh mọi `j` đứng **sau** `i`. Tổng cộng `n(n−1)/2` cặp.

### 2.2. `max()` / `min()` với hai đối số

`max(a, b)` trả số lớn hơn; dùng để "cập nhật kỷ lục": `best = max(best, candidate)`.

In [2]:
best = -1
for cand in [3, -2, 4, 1]:
    best = max(best, cand)
    print("ứng viên", cand, "-> best =", best)
print(min(7, 1), min(1, 5))

ứng viên 3 -> best = 3
ứng viên -2 -> best = 3
ứng viên 4 -> best = 4
ứng viên 1 -> best = 4
1 1


### 2.3. Giá trị nhỏ nhất **tính đến hiện tại** (running minimum)

Khi duyệt trái sang phải, biến `mn` luôn giữ **phần tử nhỏ nhất trong đoạn đã đi qua**. Đây là "bộ nhớ" duy nhất ta cần cho bài này.

In [3]:
nums = [7, 1, 5, 4, 0, 3]
mn = nums[0]
for j, x in enumerate(nums):
    mn = min(mn, x)
    print(f"j={j}, x={x}: nhỏ nhất trong nums[0..{j}] = {mn}")

j=0, x=7: nhỏ nhất trong nums[0..0] = 7
j=1, x=1: nhỏ nhất trong nums[0..1] = 1
j=2, x=5: nhỏ nhất trong nums[0..2] = 1
j=3, x=4: nhỏ nhất trong nums[0..3] = 1
j=4, x=0: nhỏ nhất trong nums[0..4] = 0
j=5, x=3: nhỏ nhất trong nums[0..5] = 0


### 2.4. `itertools.accumulate(nums, min)` — dãy prefix minimum có sẵn

`accumulate` với hàm `min` cho ra dãy `min(nums[0..0]), min(nums[0..1]), ...`.

In [4]:
from itertools import accumulate
print(list(accumulate([7, 1, 5, 4, 0, 3], min)))
print(list(accumulate([7, 1, 5, 4, 0, 3], max)))

[7, 1, 1, 1, 0, 0]
[7, 7, 7, 7, 7, 7]


### 2.5. So sánh nghiêm ngặt `<` và không nghiêm ngặt `<=`

`nums[i] < nums[j]` loại các cặp **bằng nhau**. Nếu nhầm thành `<=`, cặp bằng nhau cho hiệu 0 và kết quả có thể là `0` thay vì `-1`.

In [5]:
print(5 < 5, 5 <= 5)

False True


## 3. Cách 1 — Vét cạn mọi cặp `O(n²)`

In [6]:
def max_diff_brute(nums):
    n = len(nums)
    best = -1
    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] < nums[j]:
                best = max(best, nums[j] - nums[i])
    return best

print(max_diff_brute([7, 1, 5, 4]))    # 4
print(max_diff_brute([9, 4, 3, 2]))    # -1
print(max_diff_brute([1, 5, 2, 10]))   # 9

4
-1
9


Khởi tạo `best = -1` xử lý luôn trường hợp "không có cặp": nếu không cập nhật lần nào thì trả `-1`. Mọi hiệu hợp lệ đều `≥ 1 > -1` nên cặp hợp lệ đầu tiên luôn thay được `-1`.

### Code nộp LeetCode — Cách 1 (Accept vì n ≤ 1000)

In [7]:
from typing import List

class SolutionBrute:
    def maximumDifference(self, nums: List[int]) -> int:
        best = -1
        for i in range(len(nums)):
            for j in range(i + 1, len(nums)):
                if nums[i] < nums[j]:
                    best = max(best, nums[j] - nums[i])
        return best

print(SolutionBrute().maximumDifference([7, 1, 5, 4]))

4


## 4. Nút thắt và insight

**Nút thắt:** cố định `j`, vòng trong duyệt mọi `i < j` để tìm `nums[j] − nums[i]` lớn nhất. Nhưng với `j` cố định, hiệu lớn nhất khi **`nums[i]` nhỏ nhất**! Ta chỉ cần **một** con số: `min(nums[0..j-1])`.

**Insight:** với mỗi `j`,

$$\text{tốt nhất tại } j = nums[j] - \min(nums[0], \dots, nums[j-1])$$

(nếu `nums[j]` lớn hơn min đó). Và `min(nums[0..j-1])` có thể **cập nhật dần** khi `j` tăng (mục 2.3) → mỗi bước `O(1)`.

Bảng cho `[7, 1, 5, 4]`:

| j | nums[j] | min trước j | nums[j] − min | hợp lệ? |
|:-:|:-:|:-:|:-:|:-:|
| 1 | 1 | 7 | −6 | không |
| 2 | 5 | 1 | **4** | có |
| 3 | 4 | 1 | 3 | có |

In [8]:
nums = [7, 1, 5, 4]
for j in range(1, len(nums)):
    m = min(nums[:j])
    print(f"j={j}: nums[j] = {nums[j]}, min(nums[:{j}]) = {m}, hiệu = {nums[j] - m}")

j=1: nums[j] = 1, min(nums[:1]) = 7, hiệu = -6
j=2: nums[j] = 5, min(nums[:2]) = 1, hiệu = 4
j=3: nums[j] = 4, min(nums[:3]) = 1, hiệu = 3


Cell trên vẫn tính `min(nums[:j])` lại từ đầu mỗi lần (`O(n²)`) — chỉ để kiểm chứng ý tưởng. Lời giải chính sẽ **cập nhật** `mn` thay vì tính lại.

## 5. Cách 2 — Một lần duyệt với `mn` (lời giải chính)

Giảng từng đoạn:

```python
mn = nums[0]                    # (1) min của đoạn đã đi qua (ban đầu chỉ có nums[0])
best = -1                       # (2) chưa có cặp hợp lệ
for x in nums[1:]:              # (3) x đóng vai nums[j], j = 1..n-1
    if x > mn:                  # (4) có cặp tăng nghiêm ngặt với min
        best = max(best, x - mn)
    else:                       # (5) x <= mn: x là min mới (hoặc bằng)
        mn = x
return best
```

Tại sao `else: mn = x` đúng? Nếu `x <= mn` thì `min(mn, x) = x`. Nếu `x > mn` thì min không đổi. Gộp cả hai vào `if/else` cho gọn.

In [9]:
def max_diff(nums):
    mn = nums[0]
    best = -1
    for x in nums[1:]:
        if x > mn:
            best = max(best, x - mn)
        else:
            mn = x
    return best

print(max_diff([7, 1, 5, 4]))    # 4
print(max_diff([9, 4, 3, 2]))    # -1
print(max_diff([1, 5, 2, 10]))   # 9

4
-1
9


## 6. TRACE từng bước

Ví dụ `nums = [7, 1, 5, 4, 0, 3, 9]` — có **min mới xuất hiện muộn** (số 0) để thấy nhánh đặc biệt.

| j | x | mn trước | x > mn? | Hành động | best sau | mn sau |
|:-:|:-:|:-:|:-:|---|:-:|:-:|
| 1 | 1 | 7 | không | mn = 1 | −1 | 1 |
| 2 | 5 | 1 | có | best = max(−1, 4) | 4 | 1 |
| 3 | 4 | 1 | có | best = max(4, 3) | 4 | 1 |
| 4 | 0 | 1 | không | mn = 0 | 4 | 0 |
| 5 | 3 | 0 | có | best = max(4, 3) | 4 | 0 |
| 6 | 9 | 0 | có | best = max(4, 9) | **9** | 0 |

In [10]:
def trace_max_diff(nums):
    mn = nums[0]
    best = -1
    print(f"nums = {nums}\n  khởi tạo: mn = {mn}, best = {best}")
    for j in range(1, len(nums)):
        x = nums[j]
        if x > mn:
            old = best
            best = max(best, x - mn)
            print(f"  j={j}, x={x}: x > mn={mn} -> hiệu {x - mn}, best {old} -> {best}")
        else:
            print(f"  j={j}, x={x}: x <= mn={mn} -> mn mới = {x}")
            mn = x
    print("Kết quả:", best)
    return best

trace_max_diff([7, 1, 5, 4, 0, 3, 9])
print()
trace_max_diff([9, 4, 3, 2])

nums = [7, 1, 5, 4, 0, 3, 9]
  khởi tạo: mn = 7, best = -1
  j=1, x=1: x <= mn=7 -> mn mới = 1
  j=2, x=5: x > mn=1 -> hiệu 4, best -1 -> 4
  j=3, x=4: x > mn=1 -> hiệu 3, best 4 -> 4
  j=4, x=0: x <= mn=1 -> mn mới = 0
  j=5, x=3: x > mn=0 -> hiệu 3, best 4 -> 4
  j=6, x=9: x > mn=0 -> hiệu 9, best 4 -> 9
Kết quả: 9

nums = [9, 4, 3, 2]
  khởi tạo: mn = 9, best = -1
  j=1, x=4: x <= mn=9 -> mn mới = 4
  j=2, x=3: x <= mn=4 -> mn mới = 3
  j=3, x=2: x <= mn=3 -> mn mới = 2
Kết quả: -1


-1

Ở ví dụ thứ hai (giảm dần), nhánh `x > mn` **không bao giờ** xảy ra → `best` giữ nguyên `-1`. Đây chính là trường hợp "không tồn tại" của đề.

### Nhánh bằng nhau

In [11]:
trace_max_diff([5, 5, 5])
print()
trace_max_diff([3, 3, 4])

nums = [5, 5, 5]
  khởi tạo: mn = 5, best = -1
  j=1, x=5: x <= mn=5 -> mn mới = 5
  j=2, x=5: x <= mn=5 -> mn mới = 5
Kết quả: -1

nums = [3, 3, 4]
  khởi tạo: mn = 3, best = -1
  j=1, x=3: x <= mn=3 -> mn mới = 3
  j=2, x=4: x > mn=3 -> hiệu 1, best -1 -> 1
Kết quả: 1


1

Với `x == mn`, ta vào nhánh `else` (gán `mn = x` — không đổi gì) và **không** cập nhật `best` → `[5,5,5]` cho `-1` đúng đề.

## 7. Code nộp LeetCode

In [12]:
class Solution:
    def maximumDifference(self, nums: List[int]) -> int:
        mn = nums[0]
        best = -1
        for x in nums[1:]:
            if x > mn:
                best = max(best, x - mn)
            else:
                mn = x
        return best

for nums, expected in [([7, 1, 5, 4], 4), ([9, 4, 3, 2], -1), ([1, 5, 2, 10], 9)]:
    got = Solution().maximumDifference(nums)
    print(nums, "->", got)
    assert got == expected

[7, 1, 5, 4] -> 4
[9, 4, 3, 2] -> -1
[1, 5, 2, 10] -> 9


## 8. Cách 3 — Duyệt từ phải: giữ giá trị **lớn nhất** phía sau

Đối xứng với Cách 2: cố định `i`, hiệu lớn nhất khi `nums[j]` **lớn nhất** trong `nums[i+1..]`. Duyệt từ phải sang trái, giữ `mx` (suffix maximum).

In [13]:
def max_diff_suffix(nums):
    mx = nums[-1]
    best = -1
    for x in reversed(nums[:-1]):     # i = n-2 .. 0
        if x < mx:
            best = max(best, mx - x)
        else:
            mx = x
    return best

print(max_diff_suffix([7, 1, 5, 4]))
print(max_diff_suffix([9, 4, 3, 2]))
print(max_diff_suffix([1, 5, 2, 10]))

4
-1
9


## 9. Cách 4 — Mảng prefix minimum với `accumulate` (Pythonic)

Dựng `pre[j] = min(nums[0..j])`. Với mỗi `j ≥ 1`, ứng viên là `nums[j] − pre[j-1]` nếu dương. Viết bằng `zip`:

In [14]:
def max_diff_accumulate(nums):
    pre = list(accumulate(nums, min))
    best = max((x - m for m, x in zip(pre, nums[1:]) if x > m), default=-1)
    return best

print("pre của [7,1,5,4] =", list(accumulate([7, 1, 5, 4], min)))
print(max_diff_accumulate([7, 1, 5, 4]))
print(max_diff_accumulate([9, 4, 3, 2]))
print(max_diff_accumulate([1, 5, 2, 10]))

pre của [7,1,5,4] = [7, 1, 1, 1]
4
-1
9


`max(iterable, default=-1)`: nếu iterable **rỗng** (không có cặp tăng), `max` trả `default` thay vì ném `ValueError`. Rất hợp với yêu cầu "không có thì −1".

In [15]:
print(max([], default=-1))
try:
    max([])
except ValueError as e:
    print("không có default:", e)

-1
không có default: max() iterable argument is empty


## 10. Vì sao đúng? (invariant)

**Invariant Cách 2:** ngay trước khi xét `x = nums[j]`:
- `mn = min(nums[0..j-1])`,
- `best` = hiệu lớn nhất của các cặp hợp lệ `(i, j')` với `j' < j` (hoặc `-1` nếu chưa có).

**Duy trì:** các cặp mới có dạng `(i, j)` với `i < j`. Trong số đó, hiệu lớn nhất là `nums[j] − min(nums[0..j-1]) = x − mn`, và nó hợp lệ ⟺ `x > mn`. Nếu `x ≤ mn` thì **mọi** `nums[i] ≥ mn ≥ x` → không có cặp hợp lệ nào kết thúc ở `j`, ta chỉ cập nhật `mn = x`. Nếu `x > mn` → cập nhật `best`; `mn` không đổi vì `x > mn`.

**Kết thúc:** sau `j = n − 1`, `best` xét đủ mọi cặp → đúng là đáp án (hoặc `−1`).

## 11. Time / Space và bảng so sánh

Gọi `n = len(nums)`.

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Vét cạn | mọi cặp `i < j` | `O(n²)` | `O(1)` | Dễ nghĩ |
| 2. Running min (trái → phải) | `x − mn` | `O(n)` | `O(1)`* | **Khuyến nghị** |
| 3. Running max (phải → trái) | `mx − x` | `O(n)` | `O(1)`* | Đối xứng |
| 4. `accumulate(min)` + `zip` | mảng prefix min | `O(n)` | `O(n)` | Ngắn gọn |

\*`nums[1:]` / `reversed(nums[:-1])` tạo bản sao `O(n)`; muốn đúng `O(1)` có thể duyệt `for j in range(1, n)`.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

1. Ba ví dụ đề.
2. `n = 2`: tăng / bằng / giảm.
3. Toàn bằng nhau → `-1` (phản ví dụ cho `<=`).
4. Min ở cuối: `[5, 6, 1]` → 1.
5. Phản ví dụ "max − min toàn mảng": `[7, 1, 5, 4]` → không phải 6.
6. Giá trị lớn `10^9`.
7. Đối chiếu ngẫu nhiên giữa 4 cách.

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS ({actual})")

f = max_diff
check("đề 1", f([7, 1, 5, 4]), 4)
check("đề 2", f([9, 4, 3, 2]), -1)
check("đề 3", f([1, 5, 2, 10]), 9)
check("n=2 tăng", f([1, 2]), 1)
check("n=2 bằng", f([2, 2]), -1)
check("n=2 giảm", f([2, 1]), -1)
check("toàn bằng nhau", f([5, 5, 5, 5]), -1)
check("min ở cuối", f([5, 6, 1]), 1)
check("không phải max-min", f([7, 1, 5, 4]), 4)
check("giá trị 1e9", f([1, 10**9]), 10**9 - 1)

đề 1                         PASS (4)
đề 2                         PASS (-1)
đề 3                         PASS (9)
n=2 tăng                     PASS (1)
n=2 bằng                     PASS (-1)
n=2 giảm                     PASS (-1)
toàn bằng nhau               PASS (-1)
min ở cuối                   PASS (1)
không phải max-min           PASS (4)
giá trị 1e9                  PASS (999999999)


In [17]:
import random

rng = random.Random(2016)
for _ in range(3000):
    nums = [rng.randint(1, 10) for _ in range(rng.randint(2, 15))]
    exp = max_diff_brute(nums)
    assert max_diff(nums) == exp
    assert max_diff_suffix(nums) == exp
    assert max_diff_accumulate(nums) == exp
    assert Solution().maximumDifference(nums) == exp
print("3000 test ngẫu nhiên: 4 cách khớp vét cạn")

3000 test ngẫu nhiên: 4 cách khớp vét cạn


## 13. Lỗi người mới hay gặp

### Lỗi 1: Trả `max(nums) − min(nums)` (bỏ qua thứ tự `i < j`)

In [18]:
def wrong_max_minus_min(nums):
    d = max(nums) - min(nums)
    return d if d > 0 else -1

print("sai :", wrong_max_minus_min([7, 1, 5, 4]))   # 6
print("đúng:", max_diff([7, 1, 5, 4]))              # 4

sai : 6
đúng: 4


**Vì sao sai:** max (7) đứng **trước** min (1). Đề yêu cầu phần tử nhỏ ở **trước**. **Sửa:** running min.

### Lỗi 2: Dùng `>=` (hoặc khởi tạo `best = 0`) → trả 0 cho dãy bằng nhau

In [19]:
def wrong_best_zero(nums):
    mn, best = nums[0], 0
    for x in nums[1:]:
        best = max(best, x - mn)
        mn = min(mn, x)
    return best if best > 0 else 0      # quên -1

print("sai :", wrong_best_zero([5, 5]), wrong_best_zero([9, 4, 3, 2]))   # 0 0
print("đúng:", max_diff([5, 5]), max_diff([9, 4, 3, 2]))                 # -1 -1

sai : 0 0
đúng: -1 -1


**Vì sao sai:** đề yêu cầu `-1` khi không có cặp **tăng nghiêm ngặt**. **Sửa:** `best = -1` và chỉ cập nhật khi `x > mn`. (Đây là khác biệt so với LeetCode 121, nơi đáp án 0 là đúng.)

### Lỗi 3 (tiềm ẩn): Cập nhật `mn` **trước** khi tính hiệu

In [20]:
def wrong_update_first(nums):
    mn, best = nums[0], -1
    for x in nums:                # duyệt cả nums[0]
        mn = min(mn, x)           # cập nhật trước
        if x > mn:                # x luôn >= mn mới, x > mn không bao giờ đúng khi x là min mới
            best = max(best, x - mn)
    return best

print("kết quả:", wrong_update_first([7, 1, 5, 4]), "| đúng:", max_diff([7, 1, 5, 4]))

kết quả: 4 | đúng: 4


Ở đây kết quả **vẫn đúng** — vì khi `x` là min mới thì `x > mn` sai, còn khi không phải thì `mn` không đổi. Cập nhật trước chỉ an toàn **nhờ** điều kiện nghiêm ngặt. Nhưng nếu bỏ `if x > mn` (như LC 121: `best = max(best, x - mn)`) thì phải cẩn thận thứ tự. Thói quen tốt: **tính hiệu với min của đoạn trước, rồi mới cập nhật min**.

### Lỗi 4: Khởi tạo `mn = 0`

In [21]:
def wrong_mn_zero(nums):
    mn, best = 0, -1
    for x in nums:
        if x > mn:
            best = max(best, x - mn)
        else:
            mn = x
    return best

print("sai :", wrong_mn_zero([9, 4, 3, 2]))   # 9
print("đúng:", max_diff([9, 4, 3, 2]))        # -1

sai : 9
đúng: -1


**Vì sao sai:** `0` không phải phần tử của mảng → tạo "cặp ảo" `(0, 9)`. **Sửa:** `mn = nums[0]`.

## 14. Mẹo nhận dạng pattern

- "Max của `nums[j] − nums[i]` với `i < j`" → **running minimum** từ trái (hoặc running max từ phải).
- Tổng quát: "tối ưu `f(nums[i], nums[j])` với `i < j`" mà `f` đơn điệu theo `nums[i]` → chỉ cần giữ **giá trị tốt nhất của tiền tố**.
- Đề có "nếu không tồn tại trả −1" → khởi tạo kết quả `-1` và chỉ cập nhật khi thỏa điều kiện; hoặc `max(..., default=-1)`.

## 15. Biến thể

1. **LeetCode 121 — Best Time to Buy and Sell Stock:** giống hệt nhưng trả **0** khi không có lãi.
2. **Trả về cặp chỉ số `(i, j)`** đạt hiệu lớn nhất.
3. **LeetCode 2874 — Maximum Value of an Ordered Triplet II:** `(nums[i] − nums[j]) * nums[k]`, cũng dùng prefix max.
4. **Hiệu nhỏ nhất dương** với `i < j` — khó hơn, cần cấu trúc có thứ tự (sorted list).

In [22]:
def max_profit_121(prices):
    mn, best = prices[0], 0
    for p in prices[1:]:
        best = max(best, p - mn)
        mn = min(mn, p)
    return best

def max_diff_with_indices(nums):
    mn_i, best, pair = 0, -1, None
    for j in range(1, len(nums)):
        if nums[j] > nums[mn_i]:
            if nums[j] - nums[mn_i] > best:
                best, pair = nums[j] - nums[mn_i], (mn_i, j)
        else:
            mn_i = j
    return best, pair

print("LC 121 [7,6,4,3,1]:", max_profit_121([7, 6, 4, 3, 1]), "(LC 2016 cho -1)")
print("kèm chỉ số [7,1,5,4,0,3,9]:", max_diff_with_indices([7, 1, 5, 4, 0, 3, 9]))

LC 121 [7,6,4,3,1]: 0 (LC 2016 cho -1)
kèm chỉ số [7,1,5,4,0,3,9]: (9, (4, 6))


## 16. Bài tập tự luyện (TODO — chưa chạy)

### Bài 1 — Tính tay
Lập bảng trace (j, x, mn trước, best sau) cho `[4, 2, 8, 1, 6, 3]`.

### Bài 2 — Dự đoán output
`wrong_max_minus_min([10, 1, 2])` và `max_diff([10, 1, 2])` lần lượt bằng bao nhiêu?

### Bài 3 — Tự viết lời giải running min

In [23]:
def exercise_max_diff(nums: list[int]) -> int:
    # TODO 1: mn = nums[0], best = -1
    # TODO 2: với x trong nums[1:]: nếu x > mn thì cập nhật best, ngược lại mn = x
    pass

# assert exercise_max_diff([7, 1, 5, 4]) == 4
# assert exercise_max_diff([9, 4, 3, 2]) == -1

### Bài 4 — Viết bằng duyệt từ phải (suffix max)

In [24]:
def exercise_max_diff_suffix(nums: list[int]) -> int:
    # TODO: mx = nums[-1]; duyệt i từ n-2 về 0
    pass

# assert exercise_max_diff_suffix([1, 5, 2, 10]) == 9
# assert exercise_max_diff_suffix([5, 5]) == -1

### Bài 5 — Biến thể: hiệu **giảm** lớn nhất `nums[i] − nums[j]` với `i < j` và `nums[i] > nums[j]` (không có thì −1)

In [25]:
def exercise_max_drop(nums: list[int]) -> int:
    # TODO: giữ running MAX từ trái
    pass

# assert exercise_max_drop([7, 1, 5, 4]) == 6
# assert exercise_max_drop([1, 2, 3]) == -1

### Bài 6 — Mini challenge: LeetCode 2874 — max của `(nums[i] − nums[j]) * nums[k]` với `i < j < k`, nếu tất cả âm trả 0

In [26]:
def exercise_ordered_triplet(nums: list[int]) -> int:
    # TODO: duyệt k; giữ max_i (max nums[i] đã gặp) và max_diff (max nums[i]-nums[j] đã gặp)
    # thứ tự cập nhật: dùng max_diff cho k trước, rồi cập nhật max_diff bằng max_i - x, rồi max_i
    pass

# assert exercise_ordered_triplet([12, 6, 1, 2, 7]) == 77
# assert exercise_ordered_triplet([1, 2, 3]) == 0

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | mn bắt đầu 4 | mn đổi thành 2, rồi 1 | best sau từng bước: −1, 6, 6, 6, 6 |
| 2 | max − min toàn mảng | 10 đứng trước 1 | 9 vs 1 |
| 3 | Khuôn mục 5 | `if x > mn` | `else: mn = x` |
| 4 | `reversed(nums[:-1])` | `if x < mx` | `else: mx = x` |
| 5 | Đối xứng: giữ max | `if x < mx: best = max(best, mx - x)` | `else: mx = x` |
| 6 | Ba biến trạng thái | `ans = max(ans, max_diff * x)` | `max_diff = max(max_diff, max_i - x)`; `max_i = max(max_i, x)` |

## 18. Đáp án tham khảo, template và checklist

- Bài 1: j=1 x=2 → mn=2; j=2 x=8 → best=6; j=3 x=1 → mn=1; j=4 x=6 → 5 (best vẫn 6); j=5 x=3 → 2 → đáp án **6**.
- Bài 2: `wrong_max_minus_min([10,1,2])` = **9**; `max_diff([10,1,2])` = **1** (cặp 1 → 2).

In [27]:
def answer_max_diff(nums):
    mn, best = nums[0], -1
    for x in nums[1:]:
        if x > mn:
            best = max(best, x - mn)
        else:
            mn = x
    return best

def answer_max_diff_suffix(nums):
    mx, best = nums[-1], -1
    for i in range(len(nums) - 2, -1, -1):
        if nums[i] < mx:
            best = max(best, mx - nums[i])
        else:
            mx = nums[i]
    return best

def answer_max_drop(nums):
    mx, best = nums[0], -1
    for x in nums[1:]:
        if x < mx:
            best = max(best, mx - x)
        else:
            mx = x
    return best

def answer_ordered_triplet(nums):
    ans = max_i = max_d = 0
    for x in nums:
        ans = max(ans, max_d * x)
        max_d = max(max_d, max_i - x)
        max_i = max(max_i, x)
    return ans

assert answer_max_diff([4, 2, 8, 1, 6, 3]) == 6
assert wrong_max_minus_min([10, 1, 2]) == 9 and answer_max_diff([10, 1, 2]) == 1
assert answer_max_diff([7, 1, 5, 4]) == 4 and answer_max_diff([9, 4, 3, 2]) == -1
assert answer_max_diff_suffix([1, 5, 2, 10]) == 9 and answer_max_diff_suffix([5, 5]) == -1
assert answer_max_drop([7, 1, 5, 4]) == 6 and answer_max_drop([1, 2, 3]) == -1
assert answer_ordered_triplet([12, 6, 1, 2, 7]) == 77 and answer_ordered_triplet([1, 2, 3]) == 0
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


### Template nhớ nhanh

```python
class Solution:
    def maximumDifference(self, nums: List[int]) -> int:
        mn, best = nums[0], -1
        for x in nums[1:]:
            if x > mn:
                best = max(best, x - mn)   # cặp (min trước đó, x)
            else:
                mn = x                     # min mới
        return best
```

Câu thần chú: **"Giữ min bên trái, so với phần tử hiện tại."**

### Checklist trước khi submit
- [ ] Phần tử nhỏ phải đứng **trước** (không dùng `max − min` toàn mảng)?
- [ ] So sánh **nghiêm ngặt** `x > mn` (bằng nhau không tính)?
- [ ] `best` khởi tạo `-1` (không phải 0 như LC 121)?
- [ ] `mn` khởi tạo bằng `nums[0]` (không phải 0)?
- [ ] Đã test dãy giảm dần và dãy toàn bằng nhau?